In [1]:
# ==========================================
# 1. IMPORTS
# ==========================================
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import shutil

In [2]:
# ==========================================
# 2. PATHS
# ==========================================
# If your folder structure is: Cardiovascular > data > ptb-xl > ptbxl_database.csv
BASE_PATH = "data/ptb-xl"

# Don't join BASE_PATH if the string already includes it
CSV_PATH = os.path.join(BASE_PATH, "ptbxl_database.csv")
IMAGE_PATH = os.path.join(BASE_PATH, "images") # Assuming images are in data/ptb-xl/images

OUTPUT_PATH = "data/ecg_images"

os.makedirs(os.path.join(OUTPUT_PATH, "normal"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "abnormal"), exist_ok=True)

In [3]:
# ==========================================
# 3. LOAD METADATA (PTB-XL)
# ==========================================
# Double check the file exists before reading to avoid the crash
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("Total records:", len(df))
    print(df.head())
else:
    print(f"❌ Still can't find the file! Check this path: {os.path.abspath(CSV_PATH)}")

Total records: 21799
   ecg_id  patient_id   age  sex  height  weight  nurse  site     device  \
0       1     15709.0  56.0    1     NaN    63.0    2.0   0.0  CS-12   E   
1       2     13243.0  19.0    0     NaN    70.0    2.0   0.0  CS-12   E   
2       3     20372.0  37.0    1     NaN    69.0    2.0   0.0  CS-12   E   
3       4     17014.0  24.0    0     NaN    82.0    2.0   0.0  CS-12   E   
4       5     17448.0  19.0    1     NaN    70.0    2.0   0.0  CS-12   E   

        recording_date  ... validated_by_human  baseline_drift static_noise  \
0  1984-11-09 09:17:34  ...               True             NaN    , I-V1,     
1  1984-11-14 12:55:37  ...               True             NaN          NaN   
2  1984-11-15 12:49:10  ...               True             NaN          NaN   
3  1984-11-15 13:44:57  ...               True    , II,III,AVF          NaN   
4  1984-11-17 10:43:15  ...               True   , III,AVR,AVF          NaN   

  burst_noise electrodes_problems  extra_beats 

In [4]:
# ==========================================
# 4. AUTO LABEL (FINAL FIX)
# ==========================================

import ast

count_normal = 0
count_abnormal = 0

for i, row in df.iterrows():

    img_file = f"{row['ecg_id']}.png"
    src = os.path.join(IMAGE_PATH, img_file)

    if not os.path.exists(src):
        continue

    try:
        scp_dict = ast.literal_eval(row['scp_codes'])
    except:
        continue

    if 'NORM' in scp_dict and scp_dict['NORM'] >= 50:
        dst = os.path.join(OUTPUT_PATH, "normal", img_file)
        count_normal += 1
    else:
        dst = os.path.join(OUTPUT_PATH, "abnormal", img_file)
        count_abnormal += 1

    shutil.copy(src, dst)

print("Normal:", count_normal)
print("Abnormal:", count_abnormal)

Normal: 9438
Abnormal: 12361


In [5]:
# ==========================================
# 5. VERIFY DATA
# ==========================================

print("Normal:", count_normal)
print("Abnormal:", count_abnormal)

print("Normal samples:", len(os.listdir(f"{OUTPUT_PATH}/normal")))
print("Abnormal samples:", len(os.listdir(f"{OUTPUT_PATH}/abnormal")))

Normal: 9438
Abnormal: 12361
Normal samples: 9438
Abnormal samples: 12361


In [6]:
# ==========================================
# 6. DATA GENERATOR
# ==========================================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05
)

train_data = datagen.flow_from_directory(
    OUTPUT_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    OUTPUT_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

Found 17440 images belonging to 2 classes.
Found 4359 images belonging to 2 classes.


print(train_data.class_indices)

import numpy as np
preds = model.predict(val_data)
print("Mean prediction:", np.mean(preds))

x, y = next(train_data)
print("Sample labels:", y[:20])

In [7]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# ✅ Partial freeze (IMPORTANT)
for layer in base_model.layers[:-20]:
    layer.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

# ✅ LOW LR (CRITICAL FIX)
model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 12,077,988 (46.07 MB)

 Trainable params: 9,379,377 (35.78 MB)

 Non-trainable params: 2,698,611 (10.29 MB)

In [8]:
# ==========================================
# CLASS WEIGHTS (ADD THIS)
# ==========================================
from sklearn.utils import class_weight
import numpy as np

labels = train_data.classes

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(0.8817878450803923), 1: np.float64(1.1548139319295458)}


In [9]:
# ==========================================
# 8. TRAIN
# ==========================================

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=3, factor=0.3)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    steps_per_epoch=len(train_data),
    validation_steps=len(val_data),
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/20
545/545 ━━━━━━━━━━━━━━━━━━━━ 858s 2s/step - accuracy: 0.5225 - auc: 0.5055 - loss: 0.7088 - val_accuracy: 0.5671 - val_auc: 0.5000 - val_loss: 0.6931 - learning_rate: 5.0000e-05
Epoch 2/20
545/545 ━━━━━━━━━━━━━━━━━━━━ 709s 1s/step - accuracy: 0.5426 - auc: 0.5023 - loss: 0.6935 - val_accuracy: 0.5671 - val_auc: 0.5000 - val_loss: 0.6930 - learning_rate: 5.0000e-05
Epoch 3/20
545/545 ━━━━━━━━━━━━━━━━━━━━ 647s 1s/step - accuracy: 0.4579 - auc: 0.5001 - loss: 0.6933 - val_accuracy: 0.4370 - val_auc: 0.5000 - val_loss: 0.6932 - learning_rate: 5.0000e-05
Epoch 4/20
545/545 ━━━━━━━━━━━━━━━━━━━━ 646s 1s/step - accuracy: 0.4870 - auc: 0.5034 - loss: 0.6932 - val_accuracy: 0.4329 - val_auc: 0.5000 - val_loss: 0.6932 - learning_rate: 5.0000e-05
Epoch 5/20
545/545 ━━━━━━━━━━━━━━━━━━━━ 656s 1s/step - accuracy: 0.4534 - auc: 0.5016 - loss: 0.6933 - val_accuracy: 0.4329 - val_auc: 0.5000 - val_loss: 0.6932 - learning_rate: 5.0000e-05
Epoch 6/20
545/545 ━━━━━━━━━━━━━━━━━━━━ 1111s 2s/step -

In [10]:
from sklearn.metrics import classification_report, roc_auc_score

# Predictions
predicted_target = model.predict(val_data)
predicted_target = (predicted_target > 0.5).astype(int)

# True labels
true_target = val_data.classes

print(classification_report(true_target, predicted_target))

# ROC-AUC
auc = roc_auc_score(true_target, predicted_target)
print("ROC-AUC:", auc)

137/137 ━━━━━━━━━━━━━━━━━━━━ 122s 879ms/step
              precision    recall  f1-score   support

           0       0.57      1.00      0.72      2472
           1       0.00      0.00      0.00      1887

    accuracy                           0.57      4359
   macro avg       0.28      0.50      0.36      4359
weighted avg       0.32      0.57      0.41      4359

ROC-AUC: 0.5


c:\Users\Nisala\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Nisala\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Nisala\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [11]:
# Fine-tuning
base_model.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/20
545/545 ━━━━━━━━━━━━━━━━━━━━ 738s 1s/step - accuracy: 0.4727 - auc: 0.4988 - loss: 0.6932 - val_accuracy: 0.5132 - val_auc: 0.5000 - val_loss: 0.6931 - learning_rate: 1.0000e-05
Epoch 2/20
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4704 - auc: 0.4987 - loss: 0.6923

KeyboardInterrupt: 

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=3, factor=0.3)
]

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

In [ ]:
# ==========================================
# 9. SAVE MODEL
# ==========================================

model.save("models/ecg_model.h5")

print("✅ ECG model saved successfully!")

✅ ECG model saved successfully!


In [ ]:
# ==========================================
# 10. TEST LOAD
# ==========================================



tf.keras.models.load_model("models/ecg_model.h5")
print("✅ Model loads correctly")

✅ Model loads correctly


PTB-XL ECG Signal
        ↓
Convert to Images
        ↓
DataLoader (Step 7)
        ↓
CNN Model (Step 8)
        ↓
Training (Step 9)
        ↓
Evaluation (Step 10)
        ↓
Save Model (Step 11)